In [ ]:
print("Hello world")

In [ ]:
!unzip -q "/content/asl_data.zip" -d "/content/asl_dataset"

In [ ]:


COUNT = 500
TRAIN = '/content/asl_dataset/dataset'

Instead of ResNeXt embeddings we use plain pixel features: the 400×400 skeleton image is converted to brightness and averaged in 8×8 blocks, which gives a 50×50 = 2500-number vector per image. The same features are computed in the iOS app (`SignClassifier.swift`).

In [ ]:
from PIL import Image
import numpy as np

CANVAS = 400   # skeleton canvas, as in the dataset
GRID = 50      # 400 / 50 = 8×8 pixel blocks
BLOCK = CANVAS // GRID
OUTPUT_SIZE = GRID * GRID
LUMA = np.array([0.299, 0.587, 0.114])

def get_vec(image):
    image = image.convert('RGB')
    if image.size != (CANVAS, CANVAS):
        image = image.resize((CANVAS, CANVAS), Image.BILINEAR)
    gray = np.asarray(image, dtype=np.float64) @ LUMA / 255.0
    return gray.reshape(GRID, BLOCK, GRID, BLOCK).mean(axis=(1, 3)).ravel()

Let's load up (a sample of) our data, including pixel features and thumbnails.

In [ ]:
import arrow
import base64
import pandas as pd
from glob import iglob
from io import BytesIO
from os.path import basename
from PIL import Image

THUMBNAIL_SIZE = (64, 64)

def embed(filename: str):
    with Image.open(fp=filename, mode='r') as image:
        return get_vec(image)


# https://stackoverflow.com/a/952952
def flatten(arg):
    return [x for xs in arg for x in xs]

def png(filename: str) -> str:
    with Image.open(fp=filename, mode='r') as image:
        buffer = BytesIO()
        # our images are pretty big; let's shrink the hover images to thumbnail size
        image.resize(size=THUMBNAIL_SIZE).convert('RGB').save(buffer, format='png')
        return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()

def get_picture_from_glob(arg: str, tag: str,) -> list:
    time_get = arrow.now()
    result = [pd.Series(data=[tag, basename(input_file), embed(filename=input_file), png(filename=input_file)],
                        index=['tag', 'name', 'value', 'png'])
        for index, input_file in enumerate(list(iglob(pathname=arg))) if index < COUNT]
    print('encoded {} rows of {}  in {}'.format(len(result), tag, arrow.now() - time_get))
    return result

time_start = arrow.now()
train_dict = {basename(folder) : folder + '/*.*' for folder in iglob(TRAIN + '/*')}
train_df = pd.DataFrame(data=flatten(arg=[get_picture_from_glob(arg=value, tag=key) for key, value in train_dict.items()]))
print('done in {}'.format(arrow.now() - time_start))

Next let's use TSNE to get x/y coordinates from our pixel features.

In [ ]:
from sklearn.manifold import TSNE

train_reducer = TSNE(random_state=2025, verbose=True, n_jobs=1, perplexity=20.0, init='pca')
train_df[['x', 'y']] = train_reducer.fit_transform(X=train_df['value'].apply(func=pd.Series))

Now we're ready to plot.

In [ ]:
from bokeh.models import ColumnDataSource
from bokeh.models import HoverTool

from bokeh.plotting import figure
from bokeh.plotting import output_notebook
from bokeh.plotting import show
from bokeh.palettes import Turbo256
from bokeh.transform import factor_cmap

output_notebook()

datasource = ColumnDataSource(train_df[['png', 'tag', 'x', 'y']].sample(n=2000, random_state=2025))
factor_count = max(train_df['tag'].nunique(), 3)
indices = np.linspace(0, len(Turbo256)-1, factor_count, dtype=int)
palette = [Turbo256[index] for index in indices]
mapper = factor_cmap(field_name = 'tag', palette=palette, factors=train_df['tag'].unique().tolist(), start=0, end=factor_count-1, )

plot_figure = figure(title='TSNE projection: skeletons', width=1000, height=800, tools=('pan, wheel_zoom, reset'))

plot_figure.add_tools(HoverTool(tooltips="""
<div>
    <div>
        <img src='@png' style='float: left; margin: 5px 5px 5px 5px'/>
    </div>
    <div>
        <span style='font-size: 18px'>@tag</span>
    </div>
</div>
"""))

plot_figure.scatter(x='x', y='y', source=datasource, line_alpha=0.6, fill_alpha=0.6, size=8, color=mapper)
show(plot_figure)

What do we see? Let's build a Random Forest model and see what we get.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train_df['value'].apply(pd.Series), train_df['tag'], test_size=0.25, random_state=2025, stratify=train_df['tag'])
forest = RandomForestClassifier(n_estimators=100, random_state=2025, n_jobs=-1).fit(X_train, y_train)
print('model fit with {} trees'.format(len(forest.estimators_)))
print('accuracy: {:5.4f}'.format(accuracy_score(y_true=y_test, y_pred=forest.predict(X=X_test))))
print('f1: {:5.4f}'.format(f1_score(average='weighted', y_true=y_test, y_pred=forest.predict(X=X_test))))
print(classification_report(y_true=y_test, y_pred=forest.predict(X=X_test)))

Let's have a look at our model probabilities.

In [ ]:
import matplotlib.pyplot as plt
import warnings
from seaborn import histplot
from seaborn import scatterplot

warnings.filterwarnings('ignore', category=FutureWarning, module='seaborn')

plot_df = train_df.iloc[X_test.index][['x', 'y']].copy()
plot_df['probability'] = np.max(forest.predict_proba(X=X_test), axis=1)

plt, ax = plt.subplots(ncols=2, figsize=(12, 6))
scatterplot(ax=ax[0], data=plot_df, x='x', y='y', hue='probability', palette='viridis')
ax[0].set(xlabel=None)
ax[0].set(ylabel=None)
ax[0].set(xticklabels=[])
ax[0].set(yticklabels=[])
ax[0].tick_params(axis='both', which='both', length=0)
histplot(ax=ax[1], data=plot_df, x='probability', bins=20)

Random Forest probabilities are the share of trees voting for a class, so they are usually lower than logistic regression probabilities.

In [ ]:
import joblib; joblib.dump(forest, 'random_forest.pkl')